### This notebook compute "P5. Glacial retreat rate or index" indicator for the 27 basins of IKI Project

Spanish: Indice o tasa de retroceso glaciar

**Created:** 10/2025 by Serena Gilson. Current contact: Sophia Bakar (sbakar@rti.org)

**Assumptions:** Replaced any negative values (indicating glaciar growth, assuming that the technology just didn't pick up growth). 

**N/A v 0 Handling:** 
COMIDs with no glacier data (not in either inventory) get NaN  
COMIDs with glaciers but no loss get 0  
COMIDs with glacier loss get the calculated rate  
 
**Future work:** 
Possibly replace the glacial area with the projected shapefiles we are recieving from INAIGEM - for now, use the glacial rate as detailed in this file.

**Notes:** Could update the years with newer inventories as needed. 

Donde 1962,1955 y 2016 son los valores del contenido de masa glaciar de acuerdo al 
inventario de INAIGEM en cada año respectivamente. Cabe destacar que, en la ecuación, 
se debe considerar el año correspondiente a la imagen satelital disponible para la región
en análisis dado que, de acuerdo con el inventario de Hidrandina de 1989, las imágenes 
pueden haber sido tomadas en el año 1962 o en el año 1955. En esta ecuación, se utiliza 
el último inventario realizado en el 2018, el cual emplea imágenes satelitales del año 
2016. En caso de contar con información más actual, se recomienda utilizar ese último 
dato  

General Methodology:  
1. Calculates present and historical glacier areas by COMID.  
2. Calculates the glacier area loss rate.  
3. Uses the glacier loss rate to determine the remaining glacier area (RGl_Km2).   


In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import geopandas as gpd
from rasterstats import zonal_stats
import rasterio
import yaml
from pathlib import Path

In [2]:
# set master path for input data from config file

config_path = Path("../../config.yaml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

master_path = Path(config["master_path"])

In [3]:
# set paths for input data and indicators database 
subbasins_shapefile = master_path / "Modelacion" / "Grupos_Modelacion" / "GIS_WaterALLOC_General" / "Peru_AHD_with_districts.shp"
db_path = master_path / "Indicadores" / "BD_RiesgoClimatico_IKI.db"
glaciares_actual = master_path / "Indicadores" / "Datos_INAIGEM" / "Glaciares_2020_COMID_v2.shp"
glaciares_historico = master_path / "Indicadores" / "Datos_INAIGEM" / "Glaciares_1989_COMID_v2.shp"
glaciares_futuro_group10 = master_path / "Indicadores" / "Datos_INAIGEM" / "glaciares_2050_SSP585.shp"

In [4]:
IndID= 105 #Indicator ID (Exposure = 2 + 0X where X is the Exposure Indicator number, Peligro= 1 +0x, VSB= 3 +0x, VSS= 4 +0x, VCA= 5 +0x)
# get scenarios from database 
conn = sqlite3.connect(db_path)

scenarios_df = pd.read_sql_query(
    """
    SELECT ScnID, ScnName
    FROM ScnMod
    """,
    conn
)

conn.close()

scenario_ids = scenarios_df['ScnID'].tolist()

In [5]:
# update for additional future scenarios as needed 
scenario_glacier_config = {
    1: {
        "simulated_year": 2020,
        "use_future_glaciers": False
    },
    2: {
        "simulated_year": 2050,
        "use_future_glaciers": True # change to True later when we have a future glacier dataset
    },
    3: {
        "simulated_year": 2050,
        "use_future_glaciers": True  # change to True later when we have a future glacier dataset
    }
}

In [6]:
# define years 
actual_year= 2020
historico_year= 1962 #the inventory was processed in 1989 but the images are from 1962

# Load the shapefiles
subbasins_gdf = gpd.read_file(subbasins_shapefile).to_crs('EPSG:32718')
glaciares_actual_gdf = gpd.read_file(glaciares_actual).to_crs('EPSG:32718')
glaciares_historico_gdf = gpd.read_file(glaciares_historico).to_crs('EPSG:32718')
glaciares_futuro_gdf = gpd.read_file(glaciares_futuro_group10).to_crs('EPSG:32718')

In [7]:
# Function to calculate glacier loss and remaining area 
def compute_glacier_metrics(
    subbasins_gdf,
    glaciares_actual_gdf,
    glaciares_historico_gdf,
    simulated_year,
    use_future_glaciers=False,
    glaciares_futuro_gdf=None,
    actual_year=actual_year,
    historico_year=historico_year,
):

    actual = (
        glaciares_actual_gdf
        .groupby("COMID")["AreaKm2"]
        .sum()
        .reset_index()
        .rename(columns={"AreaKm2": "AreaKm2_Actual"})
    )

    historical = (
        glaciares_historico_gdf
        .groupby("COMID")["AreaKm2"]
        .sum()
        .reset_index()
        .rename(columns={"AreaKm2": "AreaKm2_Historical"})
    )

    merged = historical.merge(actual, on="COMID", how="outer")

    hist_area = merged["AreaKm2_Historical"].fillna(0)
    actual_area = merged["AreaKm2_Actual"].fillna(0)

    merged["RetreatRate_Baseline"] = (
        (hist_area - actual_area) / (actual_year - historico_year)
    ).clip(lower=0)

    merged["Baseline_Value"] = (
        actual_area
        - merged["RetreatRate_Baseline"] * (actual_year - actual_year)
    ).clip(lower=0)

    merged["AreaKm2_Future"] = np.nan
    merged["RetreatRate_Future"] = np.nan
    merged["Future_Value"] = np.nan

    if use_future_glaciers:

        if glaciares_futuro_gdf is None:
            raise ValueError(
                "glaciares_futuro_gdf must be provided when use_future_glaciers=True"
            )

        future = (
            glaciares_futuro_gdf
            .groupby("COMID")["Areakm2"]   # update if your field name is different
            .sum()
            .reset_index()
            .rename(columns={"Areakm2": "AreaKm2_Future"})
        )

        merged = merged.merge(
            future,
            on="COMID",
            how="outer",
            suffixes=("", "_from_future")
        )

        if "AreaKm2_Future_from_future" in merged.columns:
            merged["AreaKm2_Future"] = merged["AreaKm2_Future_from_future"]
            merged = merged.drop(columns=["AreaKm2_Future_from_future"])

        actual_area = merged["AreaKm2_Actual"].fillna(0)
        future_area = merged["AreaKm2_Future"].fillna(0)

        has_future_projection = merged["AreaKm2_Future"].notna()

        merged.loc[has_future_projection, "RetreatRate_Future"] = (
            (actual_area[has_future_projection] - future_area[has_future_projection])
            / (simulated_year - actual_year)
        ).clip(lower=0)

        merged.loc[has_future_projection, "Future_Value"] = (
            actual_area[has_future_projection]
            - merged.loc[has_future_projection, "RetreatRate_Future"]
            * (simulated_year - actual_year)
        ).clip(lower=0)

        # Use future projection only where available.
        # Everywhere else, keep baseline value.
        merged["Value"] = merged["Baseline_Value"]
        merged.loc[has_future_projection, "Value"] = merged.loc[
            has_future_projection, "Future_Value"
        ]

    else:

        merged["Value"] = merged["Baseline_Value"]

    has_glacier = (
        merged["AreaKm2_Historical"].notna()
        | merged["AreaKm2_Actual"].notna()
        | merged["AreaKm2_Future"].notna()
    )

    # COMIDs with no glacier data in any inventory get NaN
    merged.loc[~has_glacier, "Value"] = np.nan

    # COMIDs with glaciers but no remaining area stay 0
    merged.loc[has_glacier & (merged["Value"] < 0), "Value"] = 0

    merged["Value"] = merged["Value"].round(6)

    out = subbasins_gdf.merge(
        merged[
            [
                "COMID",
                "Value",
                "Baseline_Value",
                "Future_Value",
                "RetreatRate_Baseline",
                "RetreatRate_Future",
                "AreaKm2_Future",
            ]
        ],
        on="COMID",
        how="left"
    )

    return out

In [8]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

insert_query = """
INSERT OR REPLACE INTO IndValues_Dyn (ScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""

rows_to_insert = []

for scn_id in scenario_ids:

    cfg = scenario_glacier_config[scn_id]

    subbasins_with_loss = compute_glacier_metrics(
        subbasins_gdf=subbasins_gdf,
        glaciares_actual_gdf=glaciares_actual_gdf,
        glaciares_historico_gdf=glaciares_historico_gdf,
        glaciares_futuro_gdf=glaciares_futuro_gdf,
        simulated_year=cfg["simulated_year"],
        use_future_glaciers=cfg["use_future_glaciers"],
        actual_year=actual_year,
        historico_year=historico_year
    )

    for _, row in subbasins_with_loss.iterrows():

        rows_to_insert.append((
            scn_id,
            IndID,
            int(row["COMID"]),
            None if pd.isna(row["Value"]) else float(row["Value"])
        ))


In [9]:
# check that min and max values match the expected range based on the Indicators Table 
indicator_limits = pd.read_sql_query(
    """
    SELECT IndID, Min, Max
    FROM Indicators
    WHERE IndID = ?
    """,
    conn,
    params=(IndID,)
)

if indicator_limits.empty:
    raise ValueError(f"No entry found in Indicators table for IndID = {IndID}")

ind_min = indicator_limits.loc[0, 'Min']
ind_max = indicator_limits.loc[0, 'Max']

# --------------------------------------------------
# Check value ranges by scenario
# --------------------------------------------------
df_check = pd.DataFrame(
    rows_to_insert,
    columns=["ScnID", "IndID", "COMID", "Value"]
)

value_stats_by_scenario = (
    df_check
    .groupby("ScnID")["Value"]
    .agg(["min", "max", "count"])
    .reset_index()
    .merge(scenarios_df, on="ScnID", how="left")
)

value_stats_by_scenario["Indicator_Min"] = ind_min
value_stats_by_scenario["Indicator_Max"] = ind_max

print("\n=== Values to be inserted by scenario ===")
print(value_stats_by_scenario[
    [
        "ScnID",
        "ScnName",
        "min",
        "max",
        "count",
        "Indicator_Min",
        "Indicator_Max"
    ]
])

# check for duplicates
df_check = pd.DataFrame(rows_to_insert, columns=['ScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['ScnID', 'IndID', 'COMID'])
print("Duplicates in rows_to_insert:", df_check[duplicates])


=== Values to be inserted by scenario ===
   ScnID      ScnName  min        max  count  Indicator_Min  Indicator_Max
0      1   Linea Base  0.0  47.888111    330              0        50.2845
1      2  CC CMIP6 85  0.0  47.888111    330              0        50.2845
2      3  CC CMIP6 45  0.0  47.888111    330              0        50.2845
Duplicates in rows_to_insert: Empty DataFrame
Columns: [ScnID, IndID, COMID, Value]
Index: []


In [10]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()